## Задание

- [x] Выбрать датасет
- [x] Определить задачу аппроксимации
- [x] Разбить датасет на обучающую и экспериментальную выборку
- [x] Провести корреляционный анализ
- [x] Выделить 1-2 переменные, которые влияют на выход
- [x] Разбить переменные на термы (распределить равномерно по графику)
- [x] Реализовать 4 функции принадлежности и для каждой построить графики
- [x] Реализовать машину нечёткого вывода (Синглтон / Мамдани / Такаги-Сугено)

## Данные

В качестве датасета возьмём данные об использовании Инстаграм\*: [Social Media User Analysis](https://www.kaggle.com/datasets/rockyt07/social-media-user-analysis/data).
<br/>Будем аппроксимировать шкалу воспринимаемого стресса (PSS-10 или perceived stress scale).

Разделим датасет на обучающую и экспериментальную выборку в соотношении 70 к 30.

*\* соцсеть Инстаграм признана экстремистской, её деятельность запрещена на территории РФ*

## Корреляционная матрица

Начнём с вычисления корреляции между параметрами. Выберем 3 наиболее коррелирующих с параметром PSS.

In [1]:
%use coroutines
%use dataframe
%use kandy

// read dataset
val rawData = DataFrame.read("data/instagram_usage_lifestyle.csv").head(1000)
val learnData = rawData.head((rawData.rowsCount() * 0.7).toInt())
val experimentData = rawData.tail((rawData.rowsCount() * 0.3).toInt())
println("""
    Rows total: ${rawData.rowsCount()}
    Rows for learning: ${learnData.rowsCount()}
    Rows for experimental: ${experimentData.rowsCount()}
    """.trimIndent())

// evaluate correlation matrix (using pearson correlation coefficient)
val correlationMatrix = learnData.select { it.all() }.corr()
DISPLAY(correlationMatrix)

// select 3 most correlated params for PSS
val targetColumnName = "perceived_stress_score"
val correlatedColumns: List<String> =
    correlationMatrix.first {
        it["column"] == targetColumnName
    }.let { happinessRow ->
        (happinessRow.columnNames() - "column" - targetColumnName)
            .sortedBy { columnName ->
                (happinessRow[columnName] as Double).absoluteValue
            }.reversed().subList(0, 3)
    }

// show filtered correlation matrix
correlationMatrix.filter {
    it["column"] == targetColumnName
}.select("column", *correlatedColumns.toTypedArray())

Rows total: 1000
Rows for learning: 700
Rows for experimental: 300


column,user_id,age,exercise_hours_per_week,sleep_hours_per_night,perceived_stress_score,self_reported_happiness,body_mass_index,blood_pressure_systolic,blood_pressure_diastolic,daily_steps_count,weekly_work_hours,hobbies_count,social_events_per_month,books_read_per_year,volunteer_hours_per_month,travel_frequency_per_year,daily_active_minutes_instagram,sessions_per_day,posts_created_per_week,reels_watched_per_day,stories_viewed_per_day,likes_given_per_day,comments_written_per_day,dms_sent_per_week,dms_received_per_week,ads_viewed_per_day,ads_clicked_per_day,time_on_feed_per_day,time_on_explore_per_day,time_on_messages_per_day,time_on_reels_per_day,followers_count,following_count,notification_response_rate,account_creation_year,average_session_length_minutes,linked_accounts_count,user_engagement_score
user_id,"1,000000","0,038299","0,020094","-0,020590","0,052499","-0,019342","0,064498","0,006863","-0,040464","-0,017730","0,042498","-0,012476","0,077704","-0,022175","0,013747","0,062405","0,050032","0,091078","-0,014174","0,024583","0,053582","0,056536","0,055500","0,060510","0,056519","0,018557","0,033389","0,051752","0,043134","0,033427","0,046775","-0,076535","-0,052376","-0,036259","-0,018941","-0,006090","0,041596","-0,073559"
age,"0,038299","1,000000","0,029850","0,009289","-0,032667","-0,044603","0,011444","0,039451","-0,012074","0,020793","-0,008420","0,029987","0,032055","0,005551","-0,021684","-0,011634","-0,197784","-0,114723","-0,441767","-0,509868","-0,187677","-0,197621","-0,200870","-0,166111","-0,192832","-0,181848","-0,125668","-0,190456","-0,178907","-0,191185","-0,192628","-0,062579","-0,051542","0,084428","-0,041552","-0,096929","-0,064864","0,134488"
exercise_hours_per_week,"0,020094","0,029850","1,000000","0,047504","-0,004758","-0,003531","-0,003198","-0,028419","0,055073","0,041078","0,026702","0,062783","-0,072113","-0,018810","0,034787","0,036483","-0,018274","-0,045717","-0,055132","-0,028450","-0,005196","-0,016895","-0,014307","-0,009126","-0,020032","-0,011435","-0,006869","-0,025178","-0,039690","-0,039386","0,013524","0,027647","0,019249","0,014351","-0,032315","0,066521","0,044464","-0,024807"
sleep_hours_per_night,"-0,020590","0,009289","0,047504","1,000000","0,037798","-0,062524","-0,088800","0,014908","0,026433","0,058551","0,015326","0,014786","0,020190","0,074123","-0,037210","0,038542","0,059242","0,069761","0,042915","0,049259","0,057412","0,061366","0,062873","0,048194","0,056124","0,060959","0,028620","0,034491","0,087669","0,082009","0,048730","0,055108","0,062144","0,007821","-0,049693","-0,002861","-0,015702","-0,040677"
perceived_stress_score,"0,052499","-0,032667","-0,004758","0,037798","1,000000","-0,015143","0,014684","-0,033231","0,025464","-0,060623","0,092069","0,021001","-0,015867","-0,002776","-0,004290","0,019343","0,846180","0,630889","0,429634","0,689997","0,822687","0,828094","0,793227","0,778505","0,790799","0,764262","0,630777","0,830056","0,747318","0,771840","0,792903","0,056618","0,031543","-0,049181","-0,015992","0,172941","-0,050911","-0,470103"
self_reported_happiness,"-0,019342","-0,044603","-0,003531","-0,062524","-0,015143","1,000000","-0,052894","-0,034626","0,043572","-0,010632","-0,004579","-0,019821","0,074447","0,013498","-0,001441","0,015816","-0,366439","-0,276641","-0,140592","-0,317405","-0,349659","-0,359554","-0,343578","-0,309366","-0,318304","-0,315487","-0,258224","-0,342192","-0,306150","-0,335696","-0,344676","-0,038960","-0,051937","-0,040150","0,007646","-0,083891","-0,001555","0,259738"
body_mass_index,"0,064498","0,011444","-0,003198","-0,088800","0,014684","-0,052894","1,000000","-0,063357","0,003657","0,011030","0,016269","0,072599","0,032630","-0,006730","-0,032540","0,014765","0,037682","-0,005193","0,019463","0,032747","0,043551","0,035181","0,043517","0,060448","0,029269","0,024708","0,037077","0,037601","0,001433","0,029672","0,057549","0,043761","0,035716","0,014004","-0,023823","0,036934","0,003488","-0,032733"
blood_pressure_systol

column,daily_active_minutes_instagram,time_on_feed_per_day,likes_given_per_day
perceived_stress_score,"0,846180","0,830056","0,828094"


Таким образом, `daily_active_minutes_instagram` (активное время в соцсети), `likes_given_per_day` (поставленные лайки) и `stories_viewed_per_day` (количество просмотренных reels) имеют сильную положительную корреляцию c PSS.
<br/>Визуализируем на графике точки и линейную регрессию:
$$\hat{\beta} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

In [2]:
import org.jetbrains.kotlinx.kandy.ir.Plot
import org.jetbrains.kotlinx.kandy.util.color.Color

// draw linear regression
/**
 * Columns must be [Int]
 */
fun displayLinearRegression(
    columnXName: String, columnXTitle: String,
    columnYName: String, columnYTitle: String,
    limit: Int = learnData.rowsCount()
) { // y = kx + b
    val x = learnData[columnXName].cast<Int>().toList()
    val y = learnData[columnYName].cast<Int>().toList()

    val meanX = x.average()
    val meanY = y.average()

    val k = x.zip(y).sumOf { (xi, yi) ->
        (xi - meanX) * (yi - meanY)
    } / x.sumOf {
        (it - meanX).pow(2)
    }
    val b = meanY - k * meanX

    val xLine = listOf(x.min().toDouble(), x.max().toDouble())
    val yLine = xLine.map { k * it + b }

    DISPLAY(learnData.head(limit).plot {
        points {
            x(column<Double>(columnXName)) {
                axis.name = columnXTitle
            }
            y(column<Int>(columnYName)) {
                axis.name = columnYTitle
            }
            color = Color.BLUE
        }
        line {
            x(xLine)
            y(yLine)
            color = Color.RED
        }
    })
}

displayLinearRegression(
    "daily_active_minutes_instagram", "Ежедневное использование (мин)",
    targetColumnName, "PSS",
    1000
)
displayLinearRegression(
    "likes_given_per_day", "Количество поставленных лайков (в день)",
    targetColumnName, "PSS",
    1000
)
displayLinearRegression(
    "stories_viewed_per_day", "Количество просмотренных reels",
    targetColumnName, "PSS",
    1000
)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="btJb2N"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"perceived_stress_score":[3.0,1.0,4.0,18.0,19.0,1.0,23.0,33.0,12.0,25.0,14.0,15.0,7.0,7.0,31.0,2.0,37.0,40.0,20.0,12.0,7.0,6.0,10.0,25.0,22.0,14.0,30.0,0.0,30.0,26.0,3.0,22.0,22.0,27.0,10.0,23.0,3.0,12.0,22.0,1.0,32.0,10.0,19.0,26.0,4.0,19.0,7.0,2.0,17.0,39.0,35.0,35.0,32.0,11.0,34.0,26.0,18.0,4.0,27.0,17.0,30.0,26.0,19.0,6.0,8.0,0.0,38.0,29.0,29.0,15.0,31.0,8.0,11.0,27.0,30.0,21.0,25.0,33.0,38.0,12.0,36.0,5.0,30.0,2.0,10.0,35.0,0.0,6.0,39.0,6.0,8.0,20.0,14.0,1.0,17.0,26.0,1.0,36.0,11.0,24.0,11.0,40.0,24.0,34.0,40.0,32.0,10.0,12.0,30.0,40.0,20.0,28.0,34.0,28.0,23.0,36.0,8.0,26.0,12.0,23.0,32.0,21.0,14.0,9.0,14.0,22.0,15.0,31.0,16.0,5.0,26.0,30.0,29.0,38.0,22.0,22.0,35.0,34.0,19.0,5.0,8.0,27.0,15.0,26.0,28.0,10.0,25.0,17.0,18.0,13.0,28.0,27.0,19.0,34.0,30.0,24.0,34.0,14.0,9.0,34.0,5.0,6.0,19.0,39.0,7.0,12.0,20.0,40.0,33.0,2.0,31.0,34.0,20.0,9.0,8.0,10.0,19.0,26.0,36.0,8.0,23.0,16.0,25.0,37.0,1.0,5.0,6.0,23.0,24.0,28.0,30.0,8.0,31.0,31.0,38.0,36.0,12.0,35.0,0.0,16.0,19.0,22.0,40.0,7.0,16.0,25.0,27.0,40.0,27.0,8.0,11.0,23.0,9.0,35.0,28.0,31.0,32.0,38.0,29.0,29.0,0.0,24.0,20.0,7.0,14.0,38.0,17.0,24.0,28.0,0.0,28.0,30.0,6.0,9.0,15.0,37.0,28.0,5.0,30.0,13.0,25.0,2.0,38.0,11.0,6.0,18.0,31.0,11.0,31.0,33.0,26.0,9.0,4.0,15.0,0.0,18.0,24.0,29.0,9.0,18.0,37.0,14.0,25.0,0.0,9.0,2.0,7.0,31.0,26.0,26.0,29.0,32.0,3.0,36.0,22.0,18.0,36.0,34.0,28.0,3.0,33.0,32.0,36.0,37.0,25.0,3.0,38.0,12.0,8.0,11.0,11.0,33.0,0.0,19.0,21.0,19.0,6.0,2.0,33.0,5.0,22.0,26.0,9.0,4.0,8.0,6.0,39.0,28.0,10.0,23.0,22.0,4.0,10.0,17.0,8.0,9.0,10.0,35.0,9.0,30.0,5.0,11.0,36.0,5.0,5.0,23.0,21.0,14.0,38.0,13.0,29.0,2.0,22.0,14.0,12.0,6.0,6.0,32.0,34.0,26.0,15.0,13.0,13.0,40.0,6.0,4.0,39.0,28.0,13.0,33.0,36.0,9.0,17.0,9.0,22.0,12.0,38.0,20.0,33.0,1.0,22.0,4.0,6.0,13.0,6.0,31.0,23.0,35.0,33.0,7.0,30.0,11.0,2.0,38.0,40.0,13.0,17.0,19.0,26.0,0.0,4.0,20.0,1.0,36.0,16.0,18.0,39.0,1.0,35.0,33.0,37.0,12.0,38.0,40.0,36.0,8.0,5.0,36.0,36.0,28.0,35.0,2.0,26.0,31.0,28.0,4.0,36.0,16.0,25.0,13.0,12.0,12.0,32.0,9.0,40.0,19.0,34.0,19.0,31.0,8.0,25.0,34.0,4.0,35.0,24.0,1.0,8.0,14.0,36.0,34.0,8.0,0.0,38.0,10.0,33.0,28.0,21.0,8.0,31.0,18.0,38.0,27.0,18.0,31.0,15.0,11.0,6.0,7.0,25.0,16.0,8.0,17.0,35.0,26.0,16.0,7.0,11.0,19.0,29.0,13.0,18.0,27.0,36.0,6.0,3.0,26.0,24.0,30.0,39.0,14.0,4.0,37.0,8.0,10.0,18.0,2.0,17.0,8.0,39.0,11.0,13.0,5.0,32.0,35.0,28.0,34.0,2.0,22.0,34.0,11.0,1.0,9.0,21.0,23.0,37.0,24.0,22.0,40.0,35.0,36.0,13.0,29.0,1.0,35.0,13.0,7.0,4.0,35.0,13.0,21.0,8.0,19.0,17.0,26.0,33.0,24.0,28.0,23.0,3.0,24.0,34.0,26.0,34.0,40.0,34.0,4.0,21.0,24.0,31.0,5.0,39.0,1.0,30.0,2.0,23.0,21.0,26.0,10.0,29.0,17.0,29.0,30.0,12.0,10.0,33.0,37.0,20.0,35.0,34.0,12.0,11.0,22.0,23.0,0.0,40.0,20.0,38.0,11.0,9.0,40.0,39.0,33.0,31.0,24.0,23.0,36.0,10.0,21.0,22.0,37.0,2.0,8.0,16.0,31.0,26.0,28.0,18.0,7.0,19.0,31.0,33.0,35.0,13.0,28.0,16.0,27.0,20.0,5.0,39.0,26.0,8.0,22.0,13.0,5.0,0.0,0.0,6.0,15.0,21.0,38.0,26.0,19.0,39.0,15.0,0.0,32.0,7.0,29.0,22.0,7.0,37.0,16.0,38.0,11.0,21.0,15.0,29.0,28.0,40.0,27.0,10.0,36.0,14.0,16.0,25.0,28.0,5.0,40.0,12.0,26.0,24.0,37.0,35.0,18.0,2.0,23.0,34.0,11.0,28.0,35.0,2.0,9.0,33.0,16.0,1.0,36.0,39.0,31.0,10.0,29.0,20.0,12.0,17.0,29.0,36.0,35.0,22.0,38.0,12.0,36.0,7.0,40.0,37.0,1.0,25.0,3.0,13.0,2.0,21.0,33.0,11.0,2.0,8.0,8.0,13.0,2.0,35.0,21.0,5.0,21.0,34.0,36.0,26.0,32.0,34.0,8.0,37.0,5.0,23.0,30.0,29.0,38.0,14.0,31.0,17.0,17.0,33.0,5.0,6.0,7.0],
"daily_active_minutes_instagram":[5.0,74.0,5.0,233.0,184.0,49.0,125.0,282.0,128.0,196.0,240.0,123.0,5.0,5.0,363.0,5.0,273.0,473.0,224.0,96.0,141.0,5.0,187.0,221.0,59.0,213.0,224.0,5.0,329.0,241.0,67.0,2

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="hwZ2AY"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"perceived_stress_score":[3.0,1.0,4.0,18.0,19.0,1.0,23.0,33.0,12.0,25.0,14.0,15.0,7.0,7.0,31.0,2.0,37.0,40.0,20.0,12.0,7.0,6.0,10.0,25.0,22.0,14.0,30.0,0.0,30.0,26.0,3.0,22.0,22.0,27.0,10.0,23.0,3.0,12.0,22.0,1.0,32.0,10.0,19.0,26.0,4.0,19.0,7.0,2.0,17.0,39.0,35.0,35.0,32.0,11.0,34.0,26.0,18.0,4.0,27.0,17.0,30.0,26.0,19.0,6.0,8.0,0.0,38.0,29.0,29.0,15.0,31.0,8.0,11.0,27.0,30.0,21.0,25.0,33.0,38.0,12.0,36.0,5.0,30.0,2.0,10.0,35.0,0.0,6.0,39.0,6.0,8.0,20.0,14.0,1.0,17.0,26.0,1.0,36.0,11.0,24.0,11.0,40.0,24.0,34.0,40.0,32.0,10.0,12.0,30.0,40.0,20.0,28.0,34.0,28.0,23.0,36.0,8.0,26.0,12.0,23.0,32.0,21.0,14.0,9.0,14.0,22.0,15.0,31.0,16.0,5.0,26.0,30.0,29.0,38.0,22.0,22.0,35.0,34.0,19.0,5.0,8.0,27.0,15.0,26.0,28.0,10.0,25.0,17.0,18.0,13.0,28.0,27.0,19.0,34.0,30.0,24.0,34.0,14.0,9.0,34.0,5.0,6.0,19.0,39.0,7.0,12.0,20.0,40.0,33.0,2.0,31.0,34.0,20.0,9.0,8.0,10.0,19.0,26.0,36.0,8.0,23.0,16.0,25.0,37.0,1.0,5.0,6.0,23.0,24.0,28.0,30.0,8.0,31.0,31.0,38.0,36.0,12.0,35.0,0.0,16.0,19.0,22.0,40.0,7.0,16.0,25.0,27.0,40.0,27.0,8.0,11.0,23.0,9.0,35.0,28.0,31.0,32.0,38.0,29.0,29.0,0.0,24.0,20.0,7.0,14.0,38.0,17.0,24.0,28.0,0.0,28.0,30.0,6.0,9.0,15.0,37.0,28.0,5.0,30.0,13.0,25.0,2.0,38.0,11.0,6.0,18.0,31.0,11.0,31.0,33.0,26.0,9.0,4.0,15.0,0.0,18.0,24.0,29.0,9.0,18.0,37.0,14.0,25.0,0.0,9.0,2.0,7.0,31.0,26.0,26.0,29.0,32.0,3.0,36.0,22.0,18.0,36.0,34.0,28.0,3.0,33.0,32.0,36.0,37.0,25.0,3.0,38.0,12.0,8.0,11.0,11.0,33.0,0.0,19.0,21.0,19.0,6.0,2.0,33.0,5.0,22.0,26.0,9.0,4.0,8.0,6.0,39.0,28.0,10.0,23.0,22.0,4.0,10.0,17.0,8.0,9.0,10.0,35.0,9.0,30.0,5.0,11.0,36.0,5.0,5.0,23.0,21.0,14.0,38.0,13.0,29.0,2.0,22.0,14.0,12.0,6.0,6.0,32.0,34.0,26.0,15.0,13.0,13.0,40.0,6.0,4.0,39.0,28.0,13.0,33.0,36.0,9.0,17.0,9.0,22.0,12.0,38.0,20.0,33.0,1.0,22.0,4.0,6.0,13.0,6.0,31.0,23.0,35.0,33.0,7.0,30.0,11.0,2.0,38.0,40.0,13.0,17.0,19.0,26.0,0.0,4.0,20.0,1.0,36.0,16.0,18.0,39.0,1.0,35.0,33.0,37.0,12.0,38.0,40.0,36.0,8.0,5.0,36.0,36.0,28.0,35.0,2.0,26.0,31.0,28.0,4.0,36.0,16.0,25.0,13.0,12.0,12.0,32.0,9.0,40.0,19.0,34.0,19.0,31.0,8.0,25.0,34.0,4.0,35.0,24.0,1.0,8.0,14.0,36.0,34.0,8.0,0.0,38.0,10.0,33.0,28.0,21.0,8.0,31.0,18.0,38.0,27.0,18.0,31.0,15.0,11.0,6.0,7.0,25.0,16.0,8.0,17.0,35.0,26.0,16.0,7.0,11.0,19.0,29.0,13.0,18.0,27.0,36.0,6.0,3.0,26.0,24.0,30.0,39.0,14.0,4.0,37.0,8.0,10.0,18.0,2.0,17.0,8.0,39.0,11.0,13.0,5.0,32.0,35.0,28.0,34.0,2.0,22.0,34.0,11.0,1.0,9.0,21.0,23.0,37.0,24.0,22.0,40.0,35.0,36.0,13.0,29.0,1.0,35.0,13.0,7.0,4.0,35.0,13.0,21.0,8.0,19.0,17.0,26.0,33.0,24.0,28.0,23.0,3.0,24.0,34.0,26.0,34.0,40.0,34.0,4.0,21.0,24.0,31.0,5.0,39.0,1.0,30.0,2.0,23.0,21.0,26.0,10.0,29.0,17.0,29.0,30.0,12.0,10.0,33.0,37.0,20.0,35.0,34.0,12.0,11.0,22.0,23.0,0.0,40.0,20.0,38.0,11.0,9.0,40.0,39.0,33.0,31.0,24.0,23.0,36.0,10.0,21.0,22.0,37.0,2.0,8.0,16.0,31.0,26.0,28.0,18.0,7.0,19.0,31.0,33.0,35.0,13.0,28.0,16.0,27.0,20.0,5.0,39.0,26.0,8.0,22.0,13.0,5.0,0.0,0.0,6.0,15.0,21.0,38.0,26.0,19.0,39.0,15.0,0.0,32.0,7.0,29.0,22.0,7.0,37.0,16.0,38.0,11.0,21.0,15.0,29.0,28.0,40.0,27.0,10.0,36.0,14.0,16.0,25.0,28.0,5.0,40.0,12.0,26.0,24.0,37.0,35.0,18.0,2.0,23.0,34.0,11.0,28.0,35.0,2.0,9.0,33.0,16.0,1.0,36.0,39.0,31.0,10.0,29.0,20.0,12.0,17.0,29.0,36.0,35.0,22.0,38.0,12.0,36.0,7.0,40.0,37.0,1.0,25.0,3.0,13.0,2.0,21.0,33.0,11.0,2.0,8.0,8.0,13.0,2.0,35.0,21.0,5.0,21.0,34.0,36.0,26.0,32.0,34.0,8.0,37.0,5.0,23.0,30.0,29.0,38.0,14.0,31.0,17.0,17.0,33.0,5.0,6.0,7.0],
"likes_given_per_day":[28.0,68.0,25.0,132.0,103.0,55.0,80.0,158.0,90.0,131.0,139.0,78.0,18.0,32.0,184.0,31.0,145.0,271.0,150.0,70.0,99.0,29.0,102.0,134.0,59.0,137.0,148.0,32.0,188.0,130.0,46.0,136.0,88.

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="0SljvA"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"perceived_stress_score":[3.0,1.0,4.0,18.0,19.0,1.0,23.0,33.0,12.0,25.0,14.0,15.0,7.0,7.0,31.0,2.0,37.0,40.0,20.0,12.0,7.0,6.0,10.0,25.0,22.0,14.0,30.0,0.0,30.0,26.0,3.0,22.0,22.0,27.0,10.0,23.0,3.0,12.0,22.0,1.0,32.0,10.0,19.0,26.0,4.0,19.0,7.0,2.0,17.0,39.0,35.0,35.0,32.0,11.0,34.0,26.0,18.0,4.0,27.0,17.0,30.0,26.0,19.0,6.0,8.0,0.0,38.0,29.0,29.0,15.0,31.0,8.0,11.0,27.0,30.0,21.0,25.0,33.0,38.0,12.0,36.0,5.0,30.0,2.0,10.0,35.0,0.0,6.0,39.0,6.0,8.0,20.0,14.0,1.0,17.0,26.0,1.0,36.0,11.0,24.0,11.0,40.0,24.0,34.0,40.0,32.0,10.0,12.0,30.0,40.0,20.0,28.0,34.0,28.0,23.0,36.0,8.0,26.0,12.0,23.0,32.0,21.0,14.0,9.0,14.0,22.0,15.0,31.0,16.0,5.0,26.0,30.0,29.0,38.0,22.0,22.0,35.0,34.0,19.0,5.0,8.0,27.0,15.0,26.0,28.0,10.0,25.0,17.0,18.0,13.0,28.0,27.0,19.0,34.0,30.0,24.0,34.0,14.0,9.0,34.0,5.0,6.0,19.0,39.0,7.0,12.0,20.0,40.0,33.0,2.0,31.0,34.0,20.0,9.0,8.0,10.0,19.0,26.0,36.0,8.0,23.0,16.0,25.0,37.0,1.0,5.0,6.0,23.0,24.0,28.0,30.0,8.0,31.0,31.0,38.0,36.0,12.0,35.0,0.0,16.0,19.0,22.0,40.0,7.0,16.0,25.0,27.0,40.0,27.0,8.0,11.0,23.0,9.0,35.0,28.0,31.0,32.0,38.0,29.0,29.0,0.0,24.0,20.0,7.0,14.0,38.0,17.0,24.0,28.0,0.0,28.0,30.0,6.0,9.0,15.0,37.0,28.0,5.0,30.0,13.0,25.0,2.0,38.0,11.0,6.0,18.0,31.0,11.0,31.0,33.0,26.0,9.0,4.0,15.0,0.0,18.0,24.0,29.0,9.0,18.0,37.0,14.0,25.0,0.0,9.0,2.0,7.0,31.0,26.0,26.0,29.0,32.0,3.0,36.0,22.0,18.0,36.0,34.0,28.0,3.0,33.0,32.0,36.0,37.0,25.0,3.0,38.0,12.0,8.0,11.0,11.0,33.0,0.0,19.0,21.0,19.0,6.0,2.0,33.0,5.0,22.0,26.0,9.0,4.0,8.0,6.0,39.0,28.0,10.0,23.0,22.0,4.0,10.0,17.0,8.0,9.0,10.0,35.0,9.0,30.0,5.0,11.0,36.0,5.0,5.0,23.0,21.0,14.0,38.0,13.0,29.0,2.0,22.0,14.0,12.0,6.0,6.0,32.0,34.0,26.0,15.0,13.0,13.0,40.0,6.0,4.0,39.0,28.0,13.0,33.0,36.0,9.0,17.0,9.0,22.0,12.0,38.0,20.0,33.0,1.0,22.0,4.0,6.0,13.0,6.0,31.0,23.0,35.0,33.0,7.0,30.0,11.0,2.0,38.0,40.0,13.0,17.0,19.0,26.0,0.0,4.0,20.0,1.0,36.0,16.0,18.0,39.0,1.0,35.0,33.0,37.0,12.0,38.0,40.0,36.0,8.0,5.0,36.0,36.0,28.0,35.0,2.0,26.0,31.0,28.0,4.0,36.0,16.0,25.0,13.0,12.0,12.0,32.0,9.0,40.0,19.0,34.0,19.0,31.0,8.0,25.0,34.0,4.0,35.0,24.0,1.0,8.0,14.0,36.0,34.0,8.0,0.0,38.0,10.0,33.0,28.0,21.0,8.0,31.0,18.0,38.0,27.0,18.0,31.0,15.0,11.0,6.0,7.0,25.0,16.0,8.0,17.0,35.0,26.0,16.0,7.0,11.0,19.0,29.0,13.0,18.0,27.0,36.0,6.0,3.0,26.0,24.0,30.0,39.0,14.0,4.0,37.0,8.0,10.0,18.0,2.0,17.0,8.0,39.0,11.0,13.0,5.0,32.0,35.0,28.0,34.0,2.0,22.0,34.0,11.0,1.0,9.0,21.0,23.0,37.0,24.0,22.0,40.0,35.0,36.0,13.0,29.0,1.0,35.0,13.0,7.0,4.0,35.0,13.0,21.0,8.0,19.0,17.0,26.0,33.0,24.0,28.0,23.0,3.0,24.0,34.0,26.0,34.0,40.0,34.0,4.0,21.0,24.0,31.0,5.0,39.0,1.0,30.0,2.0,23.0,21.0,26.0,10.0,29.0,17.0,29.0,30.0,12.0,10.0,33.0,37.0,20.0,35.0,34.0,12.0,11.0,22.0,23.0,0.0,40.0,20.0,38.0,11.0,9.0,40.0,39.0,33.0,31.0,24.0,23.0,36.0,10.0,21.0,22.0,37.0,2.0,8.0,16.0,31.0,26.0,28.0,18.0,7.0,19.0,31.0,33.0,35.0,13.0,28.0,16.0,27.0,20.0,5.0,39.0,26.0,8.0,22.0,13.0,5.0,0.0,0.0,6.0,15.0,21.0,38.0,26.0,19.0,39.0,15.0,0.0,32.0,7.0,29.0,22.0,7.0,37.0,16.0,38.0,11.0,21.0,15.0,29.0,28.0,40.0,27.0,10.0,36.0,14.0,16.0,25.0,28.0,5.0,40.0,12.0,26.0,24.0,37.0,35.0,18.0,2.0,23.0,34.0,11.0,28.0,35.0,2.0,9.0,33.0,16.0,1.0,36.0,39.0,31.0,10.0,29.0,20.0,12.0,17.0,29.0,36.0,35.0,22.0,38.0,12.0,36.0,7.0,40.0,37.0,1.0,25.0,3.0,13.0,2.0,21.0,33.0,11.0,2.0,8.0,8.0,13.0,2.0,35.0,21.0,5.0,21.0,34.0,36.0,26.0,32.0,34.0,8.0,37.0,5.0,23.0,30.0,29.0,38.0,14.0,31.0,17.0,17.0,33.0,5.0,6.0,7.0],
"stories_viewed_per_day":[28.0,54.0,26.0,109.0,113.0,41.0,79.0,144.0,90.0,93.0,138.0,74.0,23.0,26.0,150.0,31.0,150.0,150.0,115.0,68.0,78.0,40.0,100.0,108.0,54.0,115.0,104.0,28.0,150.0,137.0,57.0,146.0,6

Распределим параметры равномерно по термам.
Ниже приведены формулы разных функций принадлежности, но мы будем использовать только треугольную:

**Треугольная**:
$$
% Треугольная (a, b, c — левая граница, вершина, правая граница)
\mu(x) = \begin{cases}
0, & x \leq a \\
\frac{x - a}{b - a}, & a < x \leq b \\
\frac{c - x}{c - b}, & b < x < c \\
0, & x \geq c
\end{cases}
$$

**Трапецеидальная**:
$$
% Трапецеидальная (a, b, c, d — границы и плато)
\mu(x) = \begin{cases}
0, & x \leq a \\
\frac{x - a}{b - a}, & a < x < b \\
1, & b \leq x \leq c \\
\frac{d - x}{d - c}, & c < x < d \\
0, & x \geq d
\end{cases}
$$

**Параболическая**:
$$
% Параболическая (a, b — границы)
\mu(x) = \begin{cases}
0, & x \leq a \\
1 - \left(\frac{x - b}{b - a}\right)^2, & a < x \leq b \\
1 - \left(\frac{x - b}{c - b}\right)^2, & b < x < c \\
0, & x \geq c
\end{cases}
$$

**Гаусса**:
$$
% Гауссова (c — центр, σ — ширина)
\mu(x) = e^{-\frac{(x - c)^2}{2\sigma^2}}
$$

In [3]:
/* API */

enum class MembershipFunctionType {
    TRIANGULAR,
    TRAPEZOIDAL,
    PARABOLIC,
    GAUSSIAN
}

data class LinguisticVariable(
    val columnName: String,
    val userFriendlyName: String,
    val membershipFunctionType: MembershipFunctionType,
    val termNames: List<String>
) {
    val minValue: Int
    val maxValue: Int

    init {
        learnData[columnName]
            .cast<Int>()
            .toList()
            .run {
                minValue = min()
                maxValue = max()
            }
    }
}

fun interface TermChartBuilder {
    /**
     * [fromX], [toX] - both inclusive
     */
    fun append(
        // input
        fromX: Double,
        toX: Double,
        termName: String,
        // output
        bufferX: MutableList<Double>,
        bufferY: MutableList<Double>,
        bufferTerm: MutableList<String>
    )
}

fun displayTerms(
    variable: LinguisticVariable,
    builder: TermChartBuilder
) {
    val step = (variable.maxValue - variable.minValue) / (variable.termNames.size.toDouble() - 1) * 2
    val fromX = variable.minValue - step / 2

    val bufferX = ArrayList<Double>()
    val bufferY = ArrayList<Double>()
    val bufferTerm = ArrayList<String>()

    variable.termNames.forEachIndexed { i, termName ->
        builder.append(
            fromX + step / 2 * i, fromX + step * (i / 2.0 + 1), termName,
            bufferX, bufferY, bufferTerm
        )
    }

    DISPLAY(learnData.plot {
        layout.title = variable.userFriendlyName
        x.axis.limits = variable.minValue.toDouble() ..  variable.maxValue.toDouble()
        y.axis.limits = 0 .. 1
        line {
            x(bufferX)
            y(bufferY)
            color(bufferTerm) { legend.name = "Терм" }
        }
    })
}

/* Term functions */

fun appendTriangularTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    bufferX += listOf(fromX, fromX + (toX - fromX) / 2.0, toX)
    bufferY += listOf(0.0, 1.0, 0.0)
    bufferTerm += List(3) { termName }
}

fun appendTrapezoidalTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    bufferX += listOf(
        fromX,
        fromX + (toX - fromX) / 3.0,
        fromX + (toX - fromX) / 3.0 * 2,
        toX
    )
    bufferY += listOf(0.0, 1.0, 1.0, 0.0)
    bufferTerm += List(4) { termName }
}

fun appendParabolicTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    val valuesX = (0 until 100).map { fromX + (toX - fromX) / 100 * it }
    val centerX = (toX - fromX) / 2 + fromX
    bufferX += valuesX
    bufferY += valuesX.map { 1 - ((it - centerX) / (centerX - fromX)).pow(2) }
    bufferTerm += List(100) { termName }
}

fun appendGaussianTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    val valuesX = (0 until 100).map { fromX + (toX - fromX) / 100 * it }
    val centerX = (toX - fromX) / 2 + fromX
    bufferX += valuesX
    bufferY += valuesX.map {
        val sigma = (centerX - fromX) / 3 // (centerX - fromX) is too big
        val numerator = - (it - centerX).pow(2)
        val denominator = 2 * sigma.pow(2)
        exp(numerator / denominator)
    }
    bufferTerm += List(100) { termName }
}

/* Impl */

val variables: List<LinguisticVariable> = listOf(
    LinguisticVariable(
        userFriendlyName = "PSS-10",
        columnName = "perceived_stress_score",
        termNames = listOf("low stress", "moderate stress", "high stress"),
        membershipFunctionType = MembershipFunctionType.TRIANGULAR
    ),

    LinguisticVariable(
        userFriendlyName = "Ежедневное использование соцсети (мин)",
        columnName = "daily_active_minutes_instagram",
        termNames = listOf("a little", "a lot", "very much", "too much"),
        membershipFunctionType = MembershipFunctionType.TRIANGULAR
    ),

    LinguisticVariable(
        userFriendlyName = "Лайков поставлено (в день)",
        columnName = "likes_given_per_day",
        termNames = listOf("a little", "a lot", "too much"),
        membershipFunctionType = MembershipFunctionType.TRIANGULAR
    ),

    LinguisticVariable(
        userFriendlyName = "Просмотрено reels",
        columnName = "stories_viewed_per_day",
        termNames = listOf("a little", "a lot", "too much"),
        membershipFunctionType = MembershipFunctionType.TRIANGULAR
    )
)

variables // display membership functions charts
    .map {
        it to TermChartBuilder(
            when (it.membershipFunctionType) {
                MembershipFunctionType.TRIANGULAR -> ::appendTriangularTerm
                MembershipFunctionType.TRAPEZOIDAL -> ::appendTrapezoidalTerm
                MembershipFunctionType.PARABOLIC -> ::appendParabolicTerm
                MembershipFunctionType.GAUSSIAN -> ::appendGaussianTerm
            }
        )
    }.toMap()
    .forEach(::displayTerms)

fun dataFrameOf(headers: List<String>, cells: List<List<*>>): DataFrame<*> =
    dataFrameOf(headers)(*cells.flatMap { it }.toTypedArray())

DISPLAY( // display table of variables
    dataFrameOf(
        headers = listOf("Переменная", "Столбец", "Минимальное значение", "Максимальное значение"),
        cells = variables.map { listOf(it.userFriendlyName, it.columnName, it.minValue, it.maxValue) }
    )
)

/* Used only by following code blocks */

val inputVariables: List<LinguisticVariable> = variables.drop(1)
val outputVariable: LinguisticVariable = variables.first()

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ajiu99"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"PSS-10"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[0.0,40.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["low stress","low stress","low stress","moderate stress","moderate stress","moderate stress","high stress","high stress","high stress"],
"x":[-20.0,0.0,20.0,0.0,20.0,40.0,20.0,40.0,60.0],
"y":[0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"11"
};
 var containerDiv = document.getElementById("ajiu99");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 PSS-10 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 low stress 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 moderate stress 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 high stress

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="cTecfJ"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Ежедневное использование соцсети (мин)"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[5.0,519.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["a little","a little","a little","a lot","a lot","a lot","very much","very much","very much","too much","too much","too much"],
"x":[-166.33333333333334,5.0,176.33333333333334,5.0,176.33333333333331,347.66666666666663,176.33333333333334,347.66666666666663,519.0,347.66666666666663,519.0,690.3333333333334],
"y":[0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"14"
};
 var containerDiv = document.getElementById("cTecfJ");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 Ежедневное использование соцсети (мин) 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 a little 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 a lot 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 very much 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 too much

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="h9vHH8"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Лайков поставлено (в день)"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[10.0,294.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["a little","a little","a little","a lot","a lot","a lot","too much","too much","too much"],
"x":[-132.0,10.0,152.0,10.0,152.0,294.0,152.0,294.0,436.0],
"y":[0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"17"
};
 var containerDiv = document.getElementById("h9vHH8");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 150 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 250 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 Лайков поставлено (в день) 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 a little 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 a lot 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 too much

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ZD00c3"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Просмотрено reels"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[21.0,150.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["a little","a little","a little","a lot","a lot","a lot","too much","too much","too much"],
"x":[-43.5,21.0,85.5,21.0,85.5,150.0,85.5,150.0,214.5],
"y":[0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"20"
};
 var containerDiv = document.getElementById("ZD00c3");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 120 
 
 
 
 
 
 
 
 
 140 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 Просмотрено reels 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 a little 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 a lot 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 too much

Переменная,Столбец,Минимальное значение,Максимальное значение
PSS-10,perceived_stress_score,0,40
Ежедневное использование соцсети (мин),daily_active_minutes_instagram,5,519
Лайков поставлено (в день),likes_given_per_day,10,294
Просмотрено reels,stories_viewed_per_day,21,150


Определим функции принадлежности, соответствующие графикам каждого терма, а также некоторый программный интерфейс над ними:

In [4]:
/* API */

fun interface AnyFunction {
    operator fun invoke(x: Double): Double
}

interface MembershipFunction : AnyFunction {
    val fromX: Double
    val toX: Double
}

infix fun AnyFunction.fuzzyOr(function: AnyFunction) = AnyFunction { max(this(it), function(it)) }
infix fun AnyFunction.fuzzyAnd(function: AnyFunction) = AnyFunction { min(this(it), function(it)) }

/* Membership functions */

fun interface MembershipFunctionBuilder {
    operator fun invoke(
        fromX: Double,
        toX: Double
    ): MembershipFunction
}

class TriangularMembershipFunction(
    override val fromX: Double,
    override val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val centerX = fromX + (toX - fromX) / 2.0
        return when {
            x <= fromX -> 0.0
            x <= centerX -> (x - fromX) / (centerX - fromX)
            x < toX -> (toX - x) / (toX - centerX)
            else -> 0.0 // x >= toX
        }
    }
}

class TrapezoidalMembershipFunction(
    override val fromX: Double,
    override val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val thirdX = fromX + (toX - fromX) / 3.0
        val twoThirdX = fromX + (toX - fromX) / 3.0 * 2
        return when {
            x <= fromX -> 0.0
            x < thirdX -> (x - fromX) / (thirdX - fromX)
            x <= twoThirdX -> 1.0
            x < toX -> (toX - x) / (toX - twoThirdX)
            else -> 0.0 // x >= toX
        }
    }
}

class ParabolicMembershipFunction(
    override val fromX: Double,
    override val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val centerX = fromX + (toX - fromX) / 2.0
        return 1 - ((x - centerX) / (centerX - fromX)).pow(2)
    }
}

class GaussianMembershipFunction(
    override val fromX: Double,
    override val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val centerX = fromX + (toX - fromX) / 2.0
        val sigma = (centerX - fromX) / 3 // (centerX - fromX) is too big
        val numerator = - (x - centerX).pow(2)
        val denominator = 2 * sigma.pow(2)
        return exp(numerator / denominator)
    }
}

fun buildEvenTerms( // distribute terms from min to max evenly
    termsCount: Int,
    minValue: Int, maxValue: Int,
    builder: MembershipFunctionBuilder
): List<MembershipFunction> {
    val step = (maxValue - minValue) / (termsCount.toDouble() - 1) * 2
    val fromX = minValue - step / 2

    return List(termsCount) { i ->
        builder(
            fromX + step / 2 * i,
            fromX + step * (i / 2 + 1)
        )
    }
}

/* Used only by following code blocks */

typealias MembershipFunctionMapping = Map<String, Map<String, MembershipFunction>>
val evenMembershipFunctionMapping: MembershipFunctionMapping =
    variables.map { variable ->
        val termsMemFuns: List<MembershipFunction> = buildEvenTerms(
            termsCount = variable.termNames.size,
            minValue = variable.minValue,
            maxValue = variable.maxValue,
            builder = MembershipFunctionBuilder(
                when (variable.membershipFunctionType) {
                    MembershipFunctionType.TRIANGULAR -> ::TriangularMembershipFunction
                    MembershipFunctionType.TRAPEZOIDAL -> ::TrapezoidalMembershipFunction
                    MembershipFunctionType.PARABOLIC -> ::ParabolicMembershipFunction
                    MembershipFunctionType.GAUSSIAN -> ::GaussianMembershipFunction
                }
            )
        )
        val termNameToFunction: Map<String, MembershipFunction> =
            variable.termNames
                .zip(termsMemFuns)
                .toMap()

        variable.columnName to termNameToFunction
    }.toMap()

operator fun MembershipFunctionMapping.get(columnName: String, termName: String): MembershipFunction =
    this[columnName]!![termName]!!

Прежде чем реализовать машину нечёткого вывода Мамдани, определим начальное приближение для набора правил на основе коэффициента степени уверенности:

In [5]:
import java.util.HashMap

typealias InputTerms = Map<String, String> // variable to term
data class ExpectedOutput(val confidenceDegree: Double, val termName: String)

// input terms (variable to term) to output term
private val intermediateRules: MutableMap<InputTerms, ExpectedOutput> = HashMap()

learnData.forEach { row ->
    data class TermAndValue(val termName: String, val memFunValue: Double)

    // variable to term
    val termsWithMaxMembership: Map<LinguisticVariable, TermAndValue> = variables.map { variable ->
        val value: Double = (row[variable.columnName] as Number).toDouble() // some cell are parsed as Int, some - as Double
        val termWithMaxMembership: TermAndValue = variable.termNames.asSequence()
            .map { termName ->
                TermAndValue(
                    termName = termName,
                    memFunValue = evenMembershipFunctionMapping[variable.columnName, termName](value)
                )
            }.maxBy(TermAndValue::memFunValue)
        variable to termWithMaxMembership
    }.toMap()

    val inputTerms: InputTerms = termsWithMaxMembership.filterKeys { variable ->
        inputVariables.contains(variable)
    }.map { (variable: LinguisticVariable, termAndValue: TermAndValue) ->
        variable.columnName to termAndValue.termName
    }.toMap()

    val confidenceDegree: Double = termsWithMaxMembership.asSequence()
        .map(Map.Entry<*, TermAndValue>::value)
        .map(TermAndValue::memFunValue)
        .reduce(Double::times)

    if (confidenceDegree > intermediateRules[inputTerms]?.confidenceDegree ?: 0.0)
        intermediateRules[inputTerms] = ExpectedOutput(
            confidenceDegree = confidenceDegree,
            termName = termsWithMaxMembership[outputVariable]!!.termName
        )
}

/* Result */

data class Rule(val inputTerms: InputTerms, val outputTerm: String) {
    override fun toString(): String =
        StringBuilder()
            .append("ЕСЛИ ")
            .apply {
                inputTerms.asSequence()
                    .flatMap { (columnName, term) ->
                        val userFriendlyName = inputVariables
                            .first { it.columnName == columnName }
                            .userFriendlyName

                        sequenceOf("\"$userFriendlyName\" = \"$term\"", " И ")
                    }.filterIndexed { i: Int, _: String ->
                        i < inputTerms.size * 2 - 1 // drop last "И"
                    }.forEach(this@apply::append)
            }.append(", ТОГДА \"${outputVariable.userFriendlyName}\" = \"$outputTerm\"")
            .toString()
}

val approximateRules: List<Rule> = intermediateRules.map { (input: InputTerms, output: ExpectedOutput) ->
    Rule(inputTerms = input, outputTerm = output.termName)
}

private val orderedInputOutputVars = inputVariables + outputVariable
dataFrameOf( // display rules as table
    headers = orderedInputOutputVars.map(LinguisticVariable::userFriendlyName),
    cells = approximateRules.map { rule ->
        inputVariables.map { rule.inputTerms[it.columnName] } + rule.outputTerm
    }
)

Ежедневное использование соцсети (мин),Лайков поставлено (в день),Просмотрено reels,PSS-10
very much,too much,too much,high stress
very much,a lot,too much,moderate stress
a lot,a little,a lot,moderate stress
very much,a lot,a lot,high stress
a little,a lot,a lot,moderate stress
a little,a little,a little,low stress
a lot,a lot,a lot,low stress
a little,a little,a lot,low stress
a lot,a lot,too much,moderate stress
too much,too much,too much,high stress


Далее, реализуем машину нечёткого вывода [Мамдани](https://docs.exponenta.ru/R2021a_nmtnew/fuzzy/types-of-fuzzy-inference-systems.html).
<br/>Алгоритм следующий:
1. Фаззификация входных значений (получить нечёткие значения).
2. Применяем операцию AND (min).
3. Применяем операцию импликации (min)
    - `A => B = not (A and not B)`.
4. Применить операцию агрегации (max).
5. Дефаззифицируем результат (ищем центроид).

Формула координаты `x` центроида (для непрерывного случая):
$$\bar{x} = \frac{\int x \cdot \mu(x) \, dx}{\int \mu(x) \, dx}$$
Используем метод прямоугольников из численного интегрирования:
$$\bar{x} = \frac{\sum_{i=1}^{n} x_i \cdot \mu(x_i)}{\sum_{i=1}^{n} \mu(x_i)}$$

In [6]:
/**
 * @param inputValues exact values (map of linguistic variable column to its value)
 */
fun fuzzifyImplicateAggregate(
    membershipFunctions: MembershipFunctionMapping,
    inputValues: Map<String, Double>,
    rules: List<Rule>
): AnyFunction =
    rules.asSequence().map { rule ->
        // apply fuzzy AND (min) to input terms
        val fuzzifiedInputs: Double = rule.inputTerms.minOf { (columnName, termName) ->
            membershipFunctions[columnName, termName](inputValues[columnName]!!)
        }
        // apply implication (min)
        AnyFunction { x ->
            min(
                fuzzifiedInputs,
                membershipFunctions[outputVariable.columnName, rule.outputTerm](x)
            )
        }
    }.reduce { f1, f2 -> f1 fuzzyOr f2 } // apply aggregation (max)

val AnyFunction.centroidX: Double get() {
    val steps = 1000
    val xValues = (0 until steps).map {
        outputVariable.minValue + (outputVariable.maxValue - outputVariable.minValue) / steps.toDouble() * it
    }

    val numerator = xValues.asSequence()
        .map { it * this(it) }
        .sum()
    val denominator = xValues.asSequence()
        .map { this(it) }
        .sum()

    return numerator / denominator // find integral using rectangles method
}

fun DataRow<*>.asInputValues(): Map<String, Double> =
    inputVariables.map { variable ->
        val variableValue = (this[variable.columnName] as Number).toDouble()
        variable.columnName to variableValue
    }.toMap()

/* Mamdani fuzzy output machine solution */

// for some rows aggregatedFun(x) = 0, so centroidX is NaN (row 449 e.g.)
private val aggregatedFun: AnyFunction = fuzzifyImplicateAggregate(
    membershipFunctions = evenMembershipFunctionMapping,
    inputValues =  experimentData[150].asInputValues(),
    rules = approximateRules
)
private val centroidX = aggregatedFun.centroidX
println("Mamdani fuzzy output machine result: $centroidX")

// visualize aggregated function and centroidX
private val steps = 1000
private val xLines: List<Double> = (0 until steps).map {
    outputVariable.minValue + (outputVariable.maxValue - outputVariable.minValue) / steps.toDouble() * it
}
private val yLines = xLines.map {
    aggregatedFun(it)
}

plot {
    layout.title = "Результат агрегации и центроид X"

    x.axis.limits = outputVariable.minValue .. outputVariable.maxValue
    y.axis.limits = 0 .. 1

    line {
        x(xLines)
        y(yLines)
        color = Color.BLUE
    }

    points {
        x(listOf(centroidX))
        y(listOf(0))
        color = Color.RED
    }
}

Mamdani fuzzy output machine result: 6.998321690493112


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="KnOfLN"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Результат агрегации и центроид X"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[0.0,40.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[0.0,0.04,0.08,0.12,0.16,0.2,0.24,0.28,0.32,0.36,0.4,0.44,0.48,0.52,0.56,0.6,0.64,0.68,0.72,0.76,0.8,0.84,0.88,0.92,0.96,1.0,1.04,1.08,1.12,1.16,1.2,1.24,1.28,1.32,1.36,1.4000000000000001,1.44,1.48,1.52,1.56,1.6,1.6400000000000001,1.68,1.72,1.76,1.8,1.84,1.8800000000000001,1.92,1.96,2.0,2.04,2.08,2.12,2.16,2.2,2.24,2.2800000000000002,2.32,2.36,2.4,2.44,2.48,2.52,2.56,2.6,2.64,2.68,2.72,2.7600000000000002,2.8000000000000003,2.84,2.88,2.92,2.96,3.0,3.04,3.08,3.12,3.16,3.2,3.24,3.2800000000000002,3.3200000000000003,3.36,3.4,3.44,3.48,3.52,3.56,3.6,3.64,3.68,3.72,3.7600000000000002,3.8000000000000003,3.84,3.88,3.92,3.96,4.0,4.04,4.08,4.12,4.16,4.2,4.24,4.28,4.32,4.36,4.4,4.44,4.48,4.5200000000000005,4.5600000000000005,4.6000000000000005,4.64,4.68,4.72,4.76,4.8,4.84,4.88,4.92,4.96,5.0,5.04,5.08,5.12,5.16,5.2,5.24,5.28,5.32,5.36,5.4,5.44,5.48,5.5200000000000005,5.5600000000000005,5.6000000000000005,5.64,5.68,5.72,5.76,5.8,5.84,5.88,5.92,5.96,6.0,6.04,6.08,6.12,6.16,6.2,6.24,6.28,6.32,6.36,6.4,6.44,6.48,6.5200000000000005,6.5600000000000005,6.6000000000000005,6.640000000000001,6.68,6.72,6.76,6.8,6.84,6.88,6.92,6.96,7.0,7.04,7.08,7.12,7.16,7.2,7.24,7.28,7.32,7.36,7.4,7.44,7.48,7.5200000000000005,7.5600000000000005,7.6000000000000005,7.640000000000001,7.68,7.72,7.76,7.8,7.84,7.88,7.92,7.96,8.0,8.040000000000001,8.08,8.120000000000001,8.16,8.2,8.24,8.28,8.32,8.36,8.4,8.44,8.48,8.52,8.56,8.6,8.64,8.68,8.72,8.76,8.8,8.84,8.88,8.92,8.96,9.0,9.040000000000001,9.08,9.120000000000001,9.16,9.200000000000001,9.24,9.28,9.32,9.36,9.4,9.44,9.48,9.52,9.56,9.6,9.64,9.68,9.72,9.76,9.8,9.84,9.88,9.92,9.96,10.0,10.040000000000001,10.08,10.120000000000001,10.16,10.200000000000001,10.24,10.28,10.32,10.36,10.4,10.44,10.48,10.52,10.56,10.6,10.64,10.68,10.72,10.76,10.8,10.84,10.88,10.92,10.96,11.0,11.040000000000001,11.08,11.120000000000001,11.16,11.200000000000001,11.24,11.28,11.32,11.36,11.4,11.44,11.48,11.52,11.56,11.6,11.64,11.68,11.72,11.76,11.8,11.84,11.88,11.92,11.96,12.0,12.040000000000001,12.08,12.120000000000001,12.16,12.200000000000001,12.24,12.280000000000001,12.32,12.36,12.4,12.44,12.48,12.52,12.56,12.6,12.64,12.68,12.72,12.76,12.8,12.84,12.88,12.92,12.96,13.0,13.040000000000001,13.08,13.120000000000001,13.16,13.200000000000001,13.24,13.280000000000001,13.32,13.36,13.4,13.44,13.48,13.52,13.56,13.6,13.64,13.68,13.72,13.76,13.8,13.84,13.88,13.92,13.96,14.0,14.040000000000001,14.08,14.120000000000001,14.16,14.200000000000001,14.24,14.280000000000001,14.32,14.36,14.4,14.44,14.48,14.52,14.56,14.6,14.64,14.68,14.72,14.76,14.8,14.84,14.88,14.92,14.96,15.0,15.040000000000001,15.08,15.120000000000001,15.16,15.200000000000001,15.24,15.280000000000001,15.32,15.36,15.4,15.44,15.48,15.52,15.56,15.6,15.64,15.68,15.72,15.76,15.8,15.84,15.88,15.92,15.96,16.0,16.04,16.080000000000002,16.12,16.16,16.2,16.240000000000002,16.28,16.32,16.36,16.4,16.44,16.48,16.52,16.56,16.6,16.64,16.68,16.72,16.76,16.8,16.84,16.88,16.92,16.96,17.0,17.04,17.080000000000002,17.12,17.16,17.2,17.240000000000002,17.28,17.32,17.36,17.400000000000002,17.44,17.48,17.52,

Оценим точность машины, пропустив через неё экспериментальную выборку и вычислив **корень среднеквадратичной ошибки**:
$$RMSE = \sqrt{\dfrac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2}$$

> RMSE (Root Mean Squared Error — корень из среднеквадратичной ошибки) — это метрика качества регрессионных моделей, показывающая среднюю величину отклонения прогнозов от фактических значений. Она измеряется в тех же единицах, что и целевая переменная, чувствительна к крупным ошибкам и чем меньше её значение, тем точнее модель.

По сути это среднеквадратичное отклонение, только относительно остатков (residual - разница между предсказанием и истинным значением), а не среднего:
$$\sigma = \sqrt{\dfrac{1}{N}\sum_{i=1}^{N}(x_i - \bar{x})^2}$$

Средняя абсолютная ошибка:
$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Средняя абсолютная процентная ошибка:
$$MAPE = \frac{1}{n} \sum_{i=1}^{n} \left| \frac{y_i - \hat{y}_i}{y_i} \right| \times 100\%$$

Область определения PSS от 0 до 40, но деление на 0 невозможно, поэтому для подобного случая используем **взвешенную абсолютную процентную ошибку**:
$$WMAPE = \frac{\sum |y_i - \hat{y}_i|}{\sum |y_i|} \times 100\%$$
или **симметричную абсолютную процентную ошибку**:
$$sMAPE = \frac{1}{n} \sum_{i=1}^{n} \frac{|y_i - \hat{y}_i|}{(|y_i| + |\hat{y}_i|) / 2} \times 100\%$$

In [7]:
import kotlin.jvm.Throws

/* API */

data class QualityMetrics(
    val mse: Double,
    val rmse: Double,
    val mae: Double,
    val wmape: Double,
    val smape: Double
) {
    fun display() {
        DISPLAY(dataFrameOf(
            headers = listOf("Метрика", "Значение"),
            cells = listOf(
                listOf("MSE", mse),
                listOf("RMSE", rmse),
                listOf("MAE", mae),
                listOf("WMAPE", "$wmape%"),
                listOf("sMAPE", "$smape%")
            )
        ))
    }
}

fun predictWithMamdaniMachine(
    membershipFunctions: MembershipFunctionMapping,
    inputValues: Map<String, Double>,
    rules: List<Rule>
): Double = fuzzifyImplicateAggregate(
    membershipFunctions = membershipFunctions,
    inputValues = inputValues,
    rules = rules
).centroidX

private val chunkSize = 5
enum class EvaluateMamdaniMachineQualityNaNBehavior {
    SKIP, // just ignore NaN centroid values, don't consider them in metrics
    ABORT // fail fast: NaNOutputException
}
class NaNOutputException(val row: DataRow<*>): Exception()
@Throws(NaNOutputException::class)
fun evaluateMamdaniMachineQuality(
    dataFrame: DataFrame<*>,
    membershipFunctions: MembershipFunctionMapping,
    rules: List<Rule>,
    nanBehavior: EvaluateMamdaniMachineQualityNaNBehavior = EvaluateMamdaniMachineQualityNaNBehavior.SKIP,
    debug: Boolean = false
): QualityMetrics {
    val trueToPredictedValues: List<Pair<Double, Double>> = runBlocking {
        val deferredTrueToPredictedValue: List<Deferred<List<Pair<Double, Double>>>> = dataFrame.asSequence()
            .chunked(chunkSize)
            .map { chunk ->
                async(context = Dispatchers.Default) {
                    val ms = System.currentTimeMillis()

                    val mappedChunk = chunk.map { row ->
                        val trueValue = (row[outputVariable.columnName] as Number).toDouble()
                        val predictedValue = predictWithMamdaniMachine(
                            membershipFunctions = membershipFunctions,
                            inputValues = row.asInputValues(),
                            rules = rules
                        )

                        trueValue to predictedValue
                    }.run {
                        when (nanBehavior) {
                            EvaluateMamdaniMachineQualityNaNBehavior.SKIP ->
                                filterNot { // missing rules can lead to NaN centroid (aggregated function = 0)
                                    it.second.isNaN() // skip such rows
                                }
                            EvaluateMamdaniMachineQualityNaNBehavior.ABORT ->
                                indexOfFirst { it.second.isNaN() }
                                    .takeUnless { it == -1 }
                                    ?.run { throw NaNOutputException(chunk[this]) }
                                    ?: this
                        }
                    }

                    if (debug) println("[${Thread.currentThread().name}] ${System.currentTimeMillis() - ms}ms")

                    mappedChunk
                }
            }.toList()

        var trueToPredictedValues = emptyList<Pair<Double, Double>>()
        for (deferredChunk in deferredTrueToPredictedValue)
            trueToPredictedValues += deferredChunk.await()

        trueToPredictedValues
    }

    val mse = trueToPredictedValues.asSequence()
        .map { (it.first - it.second).pow(2) }
        .average()

    val rmse = mse.pow(0.5)

    val mae = trueToPredictedValues.asSequence()
        .map { (it.first - it.second).absoluteValue }
        .average()

    val absTrueValuesSum = trueToPredictedValues.asSequence()
        .map(Pair<Double, *>::first)
        .map(Double::absoluteValue)
        .sum()
    val wmape = trueToPredictedValues.asSequence()
        .map { (it.first - it.second).absoluteValue }
        .sum() / absTrueValuesSum * 100

    val smape = trueToPredictedValues.asSequence()
        .map { (it.first - it.second).absoluteValue / (it.first.absoluteValue + it.second.absoluteValue) }
        .average() * 200

    return QualityMetrics(
        mse = mse,
        rmse = rmse,
        mae = mae,
        wmape = wmape,
        smape = smape
    )
}

/* Evaluate metrics */

evaluateMamdaniMachineQuality(
    dataFrame = experimentData,
    membershipFunctions = evenMembershipFunctionMapping,
    rules = approximateRules,
    debug = true // profile coroutines using `debug = true`
).display()

/* Verbose rows */

dataFrameOf(
    headers = (inputVariables + outputVariable)
        .map(LinguisticVariable::userFriendlyName)
        + "Вывод машины"
        + "Абсолютная ошибка",
    cells = experimentData.head(20).map { row ->
        val inputValuesMap = row.asInputValues()

        val inputValues = inputValuesMap.values.toList()
        val outputValue = (row[outputVariable.columnName] as Number).toDouble()
        val machineOutput = predictWithMamdaniMachine(
            membershipFunctions = evenMembershipFunctionMapping,
            inputValues = inputValuesMap,
            rules = approximateRules
        )
        val absResidual = (outputValue - machineOutput).absoluteValue

        inputValues + outputValue + machineOutput + absResidual
    }
)

[DefaultDispatcher-worker-1] 9ms
[DefaultDispatcher-worker-2] 11ms
[DefaultDispatcher-worker-3] 11ms
[DefaultDispatcher-worker-4] 11ms
[DefaultDispatcher-worker-12] 13ms
[DefaultDispatcher-worker-8] 13ms
[DefaultDispatcher-worker-5] 13ms
[DefaultDispatcher-worker-22] 16ms
[DefaultDispatcher-worker-18] 16ms
[DefaultDispatcher-worker-17] 11ms
[DefaultDispatcher-worker-11] 12ms
[DefaultDispatcher-worker-10] 21ms
[DefaultDispatcher-worker-9] 21ms
[DefaultDispatcher-worker-16] 22ms
[DefaultDispatcher-worker-29] 18ms
[DefaultDispatcher-worker-14] 19ms
[DefaultDispatcher-worker-7] 24ms
[DefaultDispatcher-worker-24] 18ms
[DefaultDispatcher-worker-21] 18ms
[DefaultDispatcher-worker-26] 15ms
[DefaultDispatcher-worker-25] 13ms
[DefaultDispatcher-worker-20] 13ms
[DefaultDispatcher-worker-19] 29ms
[DefaultDispatcher-worker-13] 29ms
[DefaultDispatcher-worker-27] 12ms
[DefaultDispatcher-worker-6] 14ms
[DefaultDispatcher-worker-15] 27ms
[DefaultDispatcher-worker-28] 20ms
[DefaultDispatcher-worker-23] 

Метрика,Значение
MSE,"93,514482"
RMSE,"9,670289"
MAE,"7,517494"
WMAPE,39.95651475532229%
sMAPE,58.30168501482538%


[DefaultDispatcher-worker-6] 23ms


Ежедневное использование соцсети (мин),Лайков поставлено (в день),Просмотрено reels,PSS-10,Вывод машины,Абсолютная ошибка
"225,000000","125,000000","128,000000","28,000000","10,000000","18,000000"
"317,000000","171,000000","140,000000","12,000000","30,643023","18,643023"
"301,000000","183,000000","143,000000","36,000000","31,036930","4,963070"
"278,000000","173,000000","135,000000","40,000000","30,709744","9,290256"
"293,000000","167,000000","149,000000","32,000000","30,508370","1,491630"
"257,000000","121,000000","125,000000","25,000000","10,000000","15,000000"
"151,000000","101,000000","88,000000","22,000000","10,000000","12,000000"
"254,000000","156,000000","141,000000","32,000000","30,130230","1,869770"
"5,000000","26,000000","33,000000","1,000000","7,128448","6,128448"
"232,000000","133,000000","115,000000","21,000000","10,000000","11,000000"


Определим функции для сохранения правил и функций принадлежности, чтобы в дальнейшем иметь возможность загрузить обученную ранее модель и продолжить её обучение.

In [8]:
private val savesDir = "saves"
private val delimiter = ','

/* Rules */

fun saveRules(rules: List<Rule>, name: String = "rules"): Unit =
    dataFrameOf( // represent as table
        headers = (inputVariables + outputVariable).map(LinguisticVariable::columnName),
        cells = rules.map { rule ->
            inputVariables.map { rule.inputTerms[it.columnName] } + rule.outputTerm
        }
    ).writeCsv("$savesDir/$name.csv", delimiter = delimiter)

fun loadRules(name: String = "rules"): List<Rule> =
    DataFrame.read("$savesDir/$name.csv")
        .rows()
        .map { row ->
            Rule(
                inputTerms = inputVariables.map(LinguisticVariable::columnName)
                    .asSequence()
                    .map { columnName ->
                        columnName to row[columnName]!!.toString()
                    }.toMap(),
                outputTerm = row[outputVariable.columnName]!!.toString()
            )
        }

/* MF's */

typealias MembershipFunctionMapping = Map<String, Map<String, MembershipFunction>>

fun saveMembershipFunctions(mapping: MembershipFunctionMapping, name: String = "functions") =
    dataFrameOf(
        headers = listOf(
            "term", // columnName/termName
            "function type",
            "from x",
            "to x"
        ),
        cells = (inputVariables + outputVariable).flatMap { variable ->
            variable.termNames.map { termName ->
                val termValue = "${variable.columnName}/$termName"
                val functionType = variable.membershipFunctionType.name
                val (fromX, toX) = when (val function = mapping[variable.columnName]!![termName]!!) {
                    is TriangularMembershipFunction -> function.fromX to function.toX
                    is TrapezoidalMembershipFunction -> function.fromX to function.toX
                    is ParabolicMembershipFunction -> function.fromX to function.toX
                    is GaussianMembershipFunction -> function.fromX to function.toX
                    else -> throw IllegalArgumentException("Couldn't save MF's: Unknown type of function")
                }
                listOf(termValue, functionType, fromX, toX)
            }
        }
    ).writeCsv("$savesDir/$name.csv", delimiter = delimiter)

fun loadMembershipFunctions(name: String = "functions"): MembershipFunctionMapping =
    DataFrame.read("$savesDir/$name.csv")
        .rows().asSequence()
        .map { row ->
            val (columnName, termName) = row["term"]!!.toString().split('/')
            val functionType = MembershipFunctionType.valueOf(row["function type"]!!.toString())
            val fromX = (row["from x"] as Number).toDouble()
            val toX = (row["to x"] as Number).toDouble()

            val pairOfTermToMf = termName to when (functionType) {
                MembershipFunctionType.TRIANGULAR -> TriangularMembershipFunction(fromX = fromX, toX = toX)
                MembershipFunctionType.TRAPEZOIDAL -> TrapezoidalMembershipFunction(fromX = fromX, toX = toX)
                MembershipFunctionType.PARABOLIC -> ParabolicMembershipFunction(fromX = fromX, toX = toX)
                MembershipFunctionType.GAUSSIAN -> GaussianMembershipFunction(fromX = fromX, toX = toX)
            }

            columnName to pairOfTermToMf
        }.groupBy(Pair<String, *>::first)
        .map { (columnName, columnToTermAndMf) -> // collect right part to map
            columnName to columnToTermAndMf.asSequence()
                .map(Pair<*, Pair<String, MembershipFunction>>::second)
                .toMap()
        }.toMap() // collect to top-level map

Далее займёмся обучением правил, используя **генетический алгоритм**.

Ключевые этапы в алгоритме:
1. Скрещивание (Crossover) - из популяции выбирается 2 родителей, чьи гены войдут в новый экземпляр.
2. Мутация - с небольшой вероятностью, часть генов может случайно измениться.
3. Приспособленность (Fitness) - формируется рейтинговая таблица в популяции, после чего худшие экземпляры отбрасываются.

**Хромосому** представим как последовательность значений термов выходной лингвистической переменной,
организованную в порядке полного перебора всех термов входных переменных.

**Генами** будет называть элементы хромосомы, то есть термы выходной переменной.

In [9]:
import java.io.FileWriter
import java.text.SimpleDateFormat
import java.util.Date
import java.util.Random // kotlin.Random doesn't have `nextGaussian()`

val random = Random(System.currentTimeMillis())

typealias InputTerms = Map<String, String>
typealias Chromosome = List<String>
typealias Population = List<Chromosome>

/* Extract chromosome from approximate rules */

private val completeInputTermsCombination: List<InputTerms> = inputVariables.fold(
    listOf<InputTerms>(emptyMap())
) { inputTermsList, inputVariable ->
    inputTermsList.flatMap { inputTerms ->
        // cross join between previously inserted values combinations and current variable values
        inputVariable.termNames.map { termName -> inputTerms + (inputVariable.columnName to termName) }
    }
}
private val initialChromosome: Chromosome = completeInputTermsCombination.map { inputTerms ->
    approximateRules.firstOrNull { rule -> rule.inputTerms == inputTerms }
        ?.outputTerm
        ?: outputVariable.termNames.run { get(random.nextInt(size)) } // fill missing rules with random terms
}

/* Operations over chromosome */

private fun Chromosome.display() = DISPLAY(dataFrameOf(
    headers = completeInputTermsCombination.first().keys.toList() + listOf(outputVariable.columnName),
    cells = completeInputTermsCombination.mapIndexed { i, inputTerms ->
        inputTerms.values.toList() + listOf(get(i))
    }
))

private infix fun Chromosome.crossover(other: Chromosome): Chromosome {
    require(size == other.size) { "Chromosomes must have equal length ($size and ${other.size} are received)" }
    return asSequence()
        .zip(other.asSequence())
        .map { (gen1, gen2) -> if (random.nextBoolean()) gen1 else gen2 }
        .toList()
}

private val mutationProbability = 0.1 // 10% chance to mutate
private fun Chromosome.mutate() = map { gen: String ->
    if (random.nextDouble() < mutationProbability)
        outputVariable.termNames.run { get(random.nextInt(size)) }
    else gen // stays the same
}

private val populationSize = 30
private fun Population.makeSelection(dataFrame: DataFrame<*>): Population =
    asSequence()
        .map { chromosome ->
            chromosome to chromosome.evaluate(dataFrame).mae // MAE is more stable (RMSE fines for rare big errors)
        }.sortedBy(Pair<*, Double>::second) // sortedBy calls lambda several times, so MAE is precalculated
        .map(Pair<Chromosome, *>::first)
        .take(populationSize)
        .toList()

fun Chromosome.evaluate(
    dataFrame: DataFrame<*>,
    debug: Boolean = false
): QualityMetrics = evaluateMamdaniMachineQuality(
    dataFrame = dataFrame,
    rules = asExemplar(),
    membershipFunctions = evenMembershipFunctionMapping,
    debug = debug
)

fun Chromosome.asExemplar(): List<Rule> =
    completeInputTermsCombination.asSequence()
        .zip(asSequence())
        .map { (inputTerms, outputTerm) -> Rule(inputTerms = inputTerms, outputTerm = outputTerm) }
        .toList()

/* Training */

private val epochs = 50
private val trainingDataframe get() = learnData

private var population: Population = List(populationSize) { initialChromosome }
private fun bestMetrics(): QualityMetrics = population.first().evaluate(trainingDataframe)

private val timestamp = SimpleDateFormat("yyyy-MM-dd_HH-mm-ss").format(Date())
private val startMs = System.currentTimeMillis()

FileWriter("logs/metrics_rules_$timestamp.csv", false).use { metricsWriter ->
    metricsWriter.appendLine("iteration,seconds,mse,rmse,mae,wmape,smape")

    for (i in 1 .. epochs) {
        // 1. crossover best one with random one
        val newbie: Chromosome = population.first() crossover population[random.nextInt(1, population.size)]

        // 2-3. mutate and make selection
        population = (
            listOf(newbie.mutate()) + population // sorting is stable, so insert to top brings changes
        ).makeSelection(trainingDataframe)

        // show metrics
        val elapsedSeconds = (System.currentTimeMillis() - startMs) / 1000

        metricsWriter.appendLine(bestMetrics().run {
            "$i,$elapsedSeconds,$mse,$rmse,$mae,$wmape,$smape"
        })
        metricsWriter.flush()
    }
}

population.first().display()

val gaLearnedRules: List<Rule> = population.first().asExemplar()

private val elapsedSeconds = (System.currentTimeMillis() - startMs) / 1000
saveRules(
    rules = gaLearnedRules,
    name = "ga_rules_e_${epochs}_p_${population.size}_r_${trainingDataframe.rowsCount()}_s_${elapsedSeconds}_t_$timestamp"
)

daily_active_minutes_instagram,likes_given_per_day,stories_viewed_per_day,perceived_stress_score
a little,a little,a little,low stress
a little,a little,a lot,low stress
a little,a little,too much,high stress
a little,a lot,a little,moderate stress
a little,a lot,a lot,moderate stress
a little,a lot,too much,moderate stress
a little,too much,a little,low stress
a little,too much,a lot,low stress
a little,too much,too much,high stress
a lot,a little,a little,low stress


Следующий шагом, используем **генетический алгоритм** для обучения параметров функций принадлежности.

In [10]:
/* Crossover (method BLX-alpha) */

private val alpha = 0.5
private infix fun Double.crossover(otherValue: Double): Double {
    if (this == otherValue)
        return this

    val d = (this - otherValue).absoluteValue
    return random.nextDouble(
        min(this, otherValue) - d * alpha,
        max(this, otherValue) + d * alpha
    )
}

data class MembershipFunctionParams(
    val center: Double,
    val width: Double,
    val min: Double,
    val max: Double
) {
    /**
     * Crossover gens
     */
    infix fun crossover(otherGen: MembershipFunctionParams) = MembershipFunctionParams(
        center = (center crossover otherGen.center).coerceIn(min, max),
        width = (width crossover otherGen.width).coerceIn(0.0, max - min),
        min = min,
        max = max
    )
}

typealias Chromosome = List<MembershipFunctionParams>
typealias Population = List<Chromosome>

infix fun Chromosome.crossover(other: Chromosome): Chromosome = asSequence()
    .zip(other.asSequence())
    .map { (c1, c2) -> c1 crossover c2 }
    .sortedBy(MembershipFunctionParams::center)
    .toList()

/* Mapping */

typealias MembershipFunctionMapping = Map<String, Map<String, MembershipFunction>>

val evenChromosome: Chromosome = evenMembershipFunctionMapping.asParams()

fun MembershipFunctionMapping.asParams(): Chromosome =
    variables.asSequence().flatMap { variable ->
        variable.termNames.map { termName ->
            get(variable.columnName, termName)
        }.map { memFun ->
            MembershipFunctionParams(
                min = variable.minValue.toDouble(),
                max = variable.maxValue.toDouble(),
                width = memFun.toX - memFun.fromX,
                center = (memFun.toX + memFun.fromX) / 2 // (memFun.toX - memFun.fromX) / 2 + memFun.fromX
            )
        }
    }.toList()

fun MembershipFunctionParams.asExemplar(functionType: MembershipFunctionType): MembershipFunction {
    val fromX = center - width / 2
    val toX = center + width / 2
    return when (functionType) {
        MembershipFunctionType.TRIANGULAR -> TriangularMembershipFunction(fromX = fromX, toX = toX)
        MembershipFunctionType.TRAPEZOIDAL -> TrapezoidalMembershipFunction(fromX = fromX, toX = toX)
        MembershipFunctionType.PARABOLIC -> ParabolicMembershipFunction(fromX = fromX, toX = toX)
        MembershipFunctionType.GAUSSIAN -> GaussianMembershipFunction(fromX = fromX, toX = toX)
    }
}

fun Chromosome.asExemplarStale(): MembershipFunctionMapping = variables.map { variable ->
    variable.columnName to variable.termNames.asSequence().zip(
        asSequence().map { it.asExemplar(functionType = variable.membershipFunctionType) } // TriangularMembershipFunction
    ).toMap()
}.toMap()

fun Chromosome.asExemplar(): MembershipFunctionMapping =
    variables.asSequence().flatMap { variable ->
        variable.termNames.map { termName -> variable to termName }
    }.zip(asSequence())
        .map { (variableToTermName, memFunParams) ->
            val (variable, termName) = variableToTermName
            val memFun = memFunParams.asExemplar(functionType = variable.membershipFunctionType)

            variable to (termName to memFun)
        }.groupBy(Pair<LinguisticVariable, *>::first)
        .map { (variable, termToFunPairs) ->
            variable.columnName to termToFunPairs
                .map(Pair<*, Pair<String, MembershipFunction>>::second)
                .toMap()
        }.toMap()

/* Mutation */

private val mutationProbability = 0.05 // 5% chance to mutate
private val sigmaFraction = 0.1 // 10% of range
private fun sigmaOf(min: Double, max: Double): Double = (max - min) * sigmaFraction
private fun Double.tryMutate(min: Double, max: Double): Double =
    if (random.nextDouble() < mutationProbability)
        (this + random.nextGaussian(
            0.0, // mean
            sigmaOf(min, max) // standard deviation
        )).coerceIn(min, max)
    else this

fun MembershipFunctionParams.mutate() = MembershipFunctionParams(
    min = min,
    max = max,
    center = center.tryMutate(min, max),
    width = width.tryMutate(0.0, max - min)
)
fun Chromosome.mutate(): Chromosome = map { it.mutate() }

/* Fitness */

val populationSize = 30
fun Population.makeSelection(
    dataFrame: DataFrame<*>,
    rules: List<Rule>
): Population =
    asSequence()
        .map { chromosome ->
            chromosome to chromosome.evaluate(dataFrame, rules).mae // MAE is more stable (RMSE fines for rare big errors)
        }.sortedBy(Pair<*, Double>::second) // sortedBy calls lambda several times, so MAE is precalculated
        .map(Pair<Chromosome, *>::first)
        .take(populationSize)
        .toList()

fun Chromosome.evaluate(
    dataFrame: DataFrame<*>,
    rules: List<Rule>,
    debug: Boolean = false
): QualityMetrics = evaluateMamdaniMachineQuality(
    dataFrame = dataFrame,
    rules = rules,
    membershipFunctions = asExemplar(),
    debug = debug
)

In [11]:
import java.io.FileWriter
import java.text.SimpleDateFormat
import java.util.Date

typealias Chromosome = List<MembershipFunctionParams>
typealias Population = List<Chromosome>

private val epochs = 100
private val trainingDataFrame get() = learnData
private val trainingRules = gaLearnedRules
private val initialChromosome = evenChromosome

var population: Population = List(populationSize) { initialChromosome }
private fun bestMetrics() = population.first().evaluate(trainingDataFrame, trainingRules)

private val timestamp = SimpleDateFormat("yyyy-MM-dd_HH-mm-ss").format(Date())
private val startMs = System.currentTimeMillis()

FileWriter("logs/metrics_functions_$timestamp.csv", false).use { metricsWriter ->
    metricsWriter.appendLine("iteration,seconds,mse,rmse,mae,wmape,smape")

    for (i in 1 .. epochs) {
        // crossover strategy: best with one of population
        val newbie: Chromosome =
            (population.first() crossover population[random.nextInt(1, populationSize)])
                .mutate()

        // fitness
        population = population.toMutableList()
            .apply { add(newbie) }
            .makeSelection(trainingDataFrame, trainingRules)

        // show metrics
        metricsWriter.appendLine(bestMetrics().run {
            val elapsedSeconds = (System.currentTimeMillis() - startMs) / 1000
            "$i,$elapsedSeconds,$mse,$rmse,$mae,$wmape,$smape"
        })
        metricsWriter.flush()
    }
}

val gaLearnedMembershipFunctions = population.first().asExemplar()

private val elapsedSeconds = (System.currentTimeMillis() - startMs) / 1000
saveMembershipFunctions(
    mapping = gaLearnedMembershipFunctions,
    name = "ga_functions_e_${epochs}_p_${population.size}_r_${trainingDataFrame.rowsCount()}_s_${elapsedSeconds}_t_$timestamp"
)

Проверим обученную через **генетический алгоритм** модель на экспериментальных данных.

In [12]:
evaluateMamdaniMachineQuality(
    dataFrame = experimentData,
    rules = gaLearnedRules,
    membershipFunctions = gaLearnedMembershipFunctions
).display()

Метрика,Значение
MSE,"62,146875"
RMSE,"7,883329"
MAE,"6,298920"
WMAPE,33.47962552396931%
sMAPE,48.97294070550126%


Следующая эвристика - **поиск по шаблону** или метод Хука - Дживса (Pattern Search).
Метод подходит для обучения параметров ФП, но слаб в обучении правил из-за дискретных значений термов.

In [40]:
typealias MembershipFunctionMapping = Map<String, Map<String, MembershipFunction>>

data class PatternSearchLearnResult(
    val epochs: Int,
    val functions: MembershipFunctionMapping
)

/**
 * @param stepPercent in 0..1; e.g. 0.5 is 50% of definition area
 * @param stepPercentMultiplier in 0..1; reduces step by this value if mapping unchanged
 * @param epsPercent in 0..1; e.g 0.2 - continue until step = 20% (0.5% leads to overfitting)
 */
fun learnMembershipFunctionsWithPatternSearch(
    initialStepPercent: Double,
    stepPercentMultiplier: Double,
    epsPercent: Double,
    trainingData: DataFrame<*>,
    trainingRules: List<Rule>,
    initialMapping: MembershipFunctionMapping
): PatternSearchLearnResult {
    fun evaluate(mapping: MembershipFunctionMapping): QualityMetrics =
        evaluateMamdaniMachineQuality(
            dataFrame = trainingData,
            membershipFunctions = mapping,
            rules = trainingRules
        )

    var stepPercent = initialStepPercent
    var currentMappingToMetrics: Pair<MembershipFunctionMapping, QualityMetrics> =
        initialMapping to evaluate(initialMapping)

    var epochs = 1

    while (stepPercent > epsPercent) {
        val currentMappingParams: List<MembershipFunctionParams> = currentMappingToMetrics.first.asParams()
        val differedMappings: List<MembershipFunctionMapping> = currentMappingParams
            .asSequence()
            .map { params ->
                val widthStep = params.width * stepPercent
                val centerStep = (params.max - params.min) * stepPercent

                sequenceOf(
                    params.copy(width = params.width + widthStep),
                    params.copy(width = params.width - widthStep),
                    params.copy(center = params.center + centerStep),
                    params.copy(center = params.center - centerStep)
                ).filter { differedParams ->
                    differedParams.width > 0 &&
                            differedParams.width <= differedParams.max - differedParams.min &&
                            differedParams.center in differedParams.min .. differedParams.max
                }
            }.mapIndexed { memFunIndex: Int, paramsVariations: Sequence<MembershipFunctionParams> -> // variations of the same MF
                paramsVariations.map { funParams ->
                    currentMappingParams.toMutableList().also {
                        it[memFunIndex] = funParams
                    }
                }.map { it.asExemplar() } // sequence of variations of MF's params to MF' sequence
            }.flatten()
            .toList()

        val bestMappingToMetrics: Pair<MembershipFunctionMapping, QualityMetrics> =
            (differedMappings
                .asSequence()
                .map { it to evaluate(it) }
                    + currentMappingToMetrics
                    ).minBy { (_, metrics) -> metrics.mae }

        val details = "epoch $epochs [MAE = ${bestMappingToMetrics.second.mae}; stepPercent = ${stepPercent * 100}%] differed mappings ${differedMappings.size}"

        if ((bestMappingToMetrics.second.mae - currentMappingToMetrics.second.mae).absoluteValue <= 0.0001) {
            stepPercent *= stepPercentMultiplier
            println("$details - mapping unchanged")
        } else {
            currentMappingToMetrics = bestMappingToMetrics
            println("$details - mapping changed")
        }

        epochs++
    }

    return PatternSearchLearnResult(epochs = epochs, functions = currentMappingToMetrics.first)
}

Оценим результат поиска по шаблону на экспериментальной выборке.

In [43]:
/* Learn */

private val trainingData = learnData
private val trainingRules = approximateRules

private val timestamp = SimpleDateFormat("yyyy-MM-dd_HH-mm-ss").format(Date())
private val startMs = System.currentTimeMillis()

private val result = learnMembershipFunctionsWithPatternSearch(
    initialStepPercent = 0.5, stepPercentMultiplier = 0.8, epsPercent = 0.2,
    trainingData = trainingData, trainingRules = trainingRules, initialMapping = evenMembershipFunctionMapping
)

/* Save */

private val elapsedSeconds = (System.currentTimeMillis() - startMs) / 1000
saveMembershipFunctions(
    mapping = result.functions,
    name = "ps_functions_e_${result.epochs}_r_${trainingData.rowsCount()}_s_${elapsedSeconds}_t_$timestamp"
)

/* Evaluate */

evaluateMamdaniMachineQuality(
    dataFrame = experimentData,
    rules = trainingRules,
    membershipFunctions = result.functions
).display()

epoch 1 [MAE = 5.599810136140559; stepPercent = 50.0%] differed mappings 33 - mapping changed
epoch 2 [MAE = 5.169995367415614; stepPercent = 50.0%] differed mappings 34 - mapping changed
epoch 3 [MAE = 4.921647264630097; stepPercent = 50.0%] differed mappings 35 - mapping changed
epoch 4 [MAE = 4.782674304729775; stepPercent = 50.0%] differed mappings 35 - mapping changed
epoch 5 [MAE = 4.674121497048618; stepPercent = 50.0%] differed mappings 35 - mapping changed
epoch 6 [MAE = 4.61925921952309; stepPercent = 50.0%] differed mappings 36 - mapping changed
epoch 7 [MAE = 4.578903552931787; stepPercent = 50.0%] differed mappings 35 - mapping changed
epoch 8 [MAE = 4.330696933409827; stepPercent = 50.0%] differed mappings 35 - mapping changed
epoch 9 [MAE = 4.29740977229931; stepPercent = 50.0%] differed mappings 36 - mapping changed
epoch 10 [MAE = 4.266266672902756; stepPercent = 50.0%] differed mappings 36 - mapping changed
epoch 11 [MAE = 4.2541868772639075; stepPercent = 50.0%] diff

Метрика,Значение
MSE,"5,779344"
RMSE,"2,404027"
MAE,"1,950753"
WMAPE,5.3506373640965155%
sMAPE,5.399132275561445%


Дообучим модель, обученную ранее на генетическом алгоритме, теперь используя поиск по шаблону.

In [44]:
/* Learn */

private val trainingData = learnData
private val trainingRules = gaLearnedRules

private val timestamp = SimpleDateFormat("yyyy-MM-dd_HH-mm-ss").format(Date())
private val startMs = System.currentTimeMillis()

private val result = learnMembershipFunctionsWithPatternSearch(
    initialStepPercent = 0.5, stepPercentMultiplier = 0.8, epsPercent = 0.2,
    trainingData = trainingData, trainingRules = trainingRules, initialMapping = gaLearnedMembershipFunctions
)

/* Save */

private val elapsedSeconds = (System.currentTimeMillis() - startMs) / 1000
saveMembershipFunctions(
    mapping = result.functions,
    name = "gaps_functions_e_${result.epochs}_r_${trainingData.rowsCount()}_s_${elapsedSeconds}_t_$timestamp"
)

/* Evaluate */

evaluateMamdaniMachineQuality(
    dataFrame = experimentData,
    rules = trainingRules,
    membershipFunctions = result.functions
).display()

epoch 1 [MAE = 5.35814491234667; stepPercent = 50.0%] differed mappings 33 - mapping changed
epoch 2 [MAE = 5.083438639601773; stepPercent = 50.0%] differed mappings 33 - mapping changed
epoch 3 [MAE = 4.828612343521889; stepPercent = 50.0%] differed mappings 34 - mapping changed
epoch 4 [MAE = 4.719502333713558; stepPercent = 50.0%] differed mappings 34 - mapping changed
epoch 5 [MAE = 4.655856070309176; stepPercent = 50.0%] differed mappings 35 - mapping changed
epoch 6 [MAE = 4.3574776960283295; stepPercent = 50.0%] differed mappings 36 - mapping changed
epoch 7 [MAE = 4.300119324194944; stepPercent = 50.0%] differed mappings 36 - mapping changed
epoch 8 [MAE = 4.284622237290033; stepPercent = 50.0%] differed mappings 35 - mapping changed
epoch 9 [MAE = 4.251267808949572; stepPercent = 50.0%] differed mappings 35 - mapping changed
epoch 10 [MAE = 4.251267808949572; stepPercent = 50.0%] differed mappings 35 - mapping unchanged
epoch 11 [MAE = 4.25079428304659; stepPercent = 40.0%] di

Метрика,Значение
MSE,"19,430865"
RMSE,"4,408045"
MAE,"3,546669"
WMAPE,11.020723193992874%
sMAPE,16.355806179771275%
